<a href="https://colab.research.google.com/github/sabyapaul/AgenticAI-Lab/blob/main/Gradio_Agentic_AI_Bronze_to_Silver_Eligibility_Validation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Step 1: Install packages**

In [1]:
!pip install pyspark gradio -q

**Step 2: Copy this full code into one Colab cell**

In [2]:
import gradio as gr
import pandas as pd
from datetime import datetime

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, lit, current_timestamp

spark = SparkSession.builder \
    .appName("Agentic_AI_Eligibility_Validation_Demo") \
    .getOrCreate()


def run_eligibility_agent():

    # -----------------------------
    # 1. Bronze sample claims data
    # -----------------------------
    claims_data = [
        ("C001", "M001", "2026-01-15", 1200.00),
        ("C002", "M002", "2026-02-10", 800.00),
        ("C003", "M003", "2026-03-05", 500.00),
        ("C004", "M004", "2026-01-20", 700.00),
        ("C005", "M001", "2026-04-10", 300.00)
    ]

    bronze_claim = spark.createDataFrame(
        claims_data,
        ["claim_id", "member_id", "service_date", "claim_amount"]
    ).withColumn("service_date", col("service_date").cast("date"))

    # -----------------------------
    # 2. Bronze eligibility data
    # -----------------------------
    eligibility_data = [
        ("M001", "2026-01-01", "2026-03-31"),
        ("M002", "2026-01-01", "2026-12-31"),
        ("M003", "2026-04-01", "2026-12-31")
    ]

    bronze_eligibility = spark.createDataFrame(
        eligibility_data,
        ["member_id", "effective_date", "termination_date"]
    ).withColumn(
        "effective_date", col("effective_date").cast("date")
    ).withColumn(
        "termination_date", col("termination_date").cast("date")
    )

    # -----------------------------
    # 3. Bronze to Silver staging
    # -----------------------------
    silver_claim = bronze_claim
    silver_eligibility = bronze_eligibility

    silver_claim.createOrReplaceTempView("silver_claim")
    silver_eligibility.createOrReplaceTempView("silver_eligibility")

    # -----------------------------
    # 4. Agent validation rule
    # -----------------------------
    invalid_claims = spark.sql("""
        SELECT c.claim_id, c.member_id, c.service_date, c.claim_amount
        FROM silver_claim c
        LEFT JOIN silver_eligibility e
          ON c.member_id = e.member_id
         AND c.service_date BETWEEN e.effective_date AND e.termination_date
        WHERE e.member_id IS NULL
    """)

    valid_claims = spark.sql("""
        SELECT c.claim_id, c.member_id, c.service_date, c.claim_amount
        FROM silver_claim c
        INNER JOIN silver_eligibility e
          ON c.member_id = e.member_id
         AND c.service_date BETWEEN e.effective_date AND e.termination_date
    """)

    # -----------------------------
    # 5. Agent takes action
    # -----------------------------
    flagged_claims = invalid_claims \
        .withColumn("validation_status", lit("FAILED")) \
        .withColumn("reason_code", lit("NO_ACTIVE_ELIGIBILITY")) \
        .withColumn("agent_action", lit("FLAG_FOR_REVIEW")) \
        .withColumn("agent_name", lit("Eligibility Validation Agent"))

    passed_claims = valid_claims \
        .withColumn("validation_status", lit("PASSED")) \
        .withColumn("reason_code", lit("")) \
        .withColumn("agent_action", lit("LOAD_TO_SILVER")) \
        .withColumn("agent_name", lit("Eligibility Validation Agent"))

    final_df = passed_claims.unionByName(flagged_claims)

    # Convert Spark to Pandas for Gradio
    final_pdf = final_df.toPandas()
    exception_pdf = flagged_claims.toPandas()

    total_claims = final_pdf.shape[0]
    passed_count = final_pdf[final_pdf["validation_status"] == "PASSED"].shape[0]
    failed_count = final_pdf[final_pdf["validation_status"] == "FAILED"].shape[0]

    summary = f"""
# Eligibility Validation Agent Summary

**Business Rule:** Claim service date must fall within member eligibility period.

## Result
- Total Claims Processed: **{total_claims}**
- Claims Loaded to Silver: **{passed_count}**
- Claims Flagged for Review: **{failed_count}**

## Agent Decision
The agent checked each claim service date against member eligibility effective and termination dates.

## Action Taken
- Valid claims were moved to the Silver layer.
- Invalid claims were flagged with `NO_ACTIVE_ELIGIBILITY`.
- Failed claims were routed to the exception queue.
"""

    reasoning = """
## Agent Reasoning Example

### Claim C003
- Member: M003
- Service Date: 2026-03-05
- Eligibility Start Date: 2026-04-01
- Decision: Failed
- Reason: Service date occurred before eligibility start date.
- Action: FLAG_FOR_REVIEW

### Claim C004
- Member: M004
- Eligibility Record: Not Found
- Decision: Failed
- Reason: No active eligibility record exists.
- Action: FLAG_FOR_REVIEW

### Claim C005
- Member: M001
- Service Date: 2026-04-10
- Eligibility End Date: 2026-03-31
- Decision: Failed
- Reason: Service date occurred after eligibility termination date.
- Action: FLAG_FOR_REVIEW
"""

    audit_trail = f"""
{datetime.now().strftime('%Y-%m-%d %H:%M:%S')} - Eligibility Validation Agent started
{datetime.now().strftime('%Y-%m-%d %H:%M:%S')} - Bronze claims loaded
{datetime.now().strftime('%Y-%m-%d %H:%M:%S')} - Bronze eligibility loaded
{datetime.now().strftime('%Y-%m-%d %H:%M:%S')} - Business rule executed
{datetime.now().strftime('%Y-%m-%d %H:%M:%S')} - {passed_count} claims loaded to Silver
{datetime.now().strftime('%Y-%m-%d %H:%M:%S')} - {failed_count} claims flagged for review
{datetime.now().strftime('%Y-%m-%d %H:%M:%S')} - Exception queue created
"""

    return summary, final_pdf, exception_pdf, reasoning, audit_trail


# -----------------------------
# Gradio UI
# -----------------------------
with gr.Blocks(title="Agentic AI Eligibility Validation Demo") as demo:

    gr.Markdown("""
    # Agentic AI Demo: Eligibility Validation Agent

    **Use Case:** Bronze to Silver Data Quality Validation for Healthcare Claims
    **Rule:** Claim service date must fall within member eligibility period.
    """)

    run_button = gr.Button("Run Eligibility Validation Agent")

    summary_output = gr.Markdown(label="Agent Summary")

    final_table = gr.Dataframe(
        label="Final Silver Claims Output",
        interactive=False
    )

    exception_table = gr.Dataframe(
        label="Exception Queue",
        interactive=False
    )

    reasoning_output = gr.Markdown(label="Agent Reasoning")

    audit_output = gr.Textbox(
        label="Agent Audit Trail",
        lines=10
    )

    run_button.click(
        fn=run_eligibility_agent,
        inputs=[],
        outputs=[
            summary_output,
            final_table,
            exception_table,
            reasoning_output,
            audit_output
        ]
    )

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://43ad7b8e0ed700873a.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
